# Phase 9: Transformers & Attention

## What you'll learn
- Scaled dot-product attention from scratch
- Multi-head attention
- Positional encoding
- `nn.Transformer` module
- Build a mini GPT-style model
- Hugging Face integration

---

## Why Transformers?

Transformers replaced RNNs for most sequence tasks because they:
- Process all positions **in parallel** (no sequential bottleneck)
- Capture **long-range dependencies** via attention
- Scale better with data and compute

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 9.1 Scaled Dot-Product Attention (From Scratch)

The core operation: **Attention(Q, K, V) = softmax(QK^T / √d_k) V**

- **Q** (Query): what am I looking for?
- **K** (Key): what do I contain?
- **V** (Value): what information do I provide?
- Scale by √d_k to prevent softmax saturation

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights

# Test
seq_len, d_model = 5, 8
Q = K = V = torch.randn(1, seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output: {output.shape}")   # (1, 5, 8)
print(f"Weights: {weights.shape}") # (1, 5, 5)
print(f"Weights sum per row: {weights.sum(dim=-1)}")  # all 1.0

## 9.2 Causal Mask (for Autoregressive Models)

In GPT-style models, each position can only attend to **previous** positions.

In [ ]:
def create_causal_mask(seq_len):
    """Lower triangular mask: position i can attend to positions 0..i"""
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)

mask = create_causal_mask(5)
print(f"Causal mask:\n{mask[0]}")

# With mask
output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
print(f"\nMasked attention weights:\n{weights[0].detach()}")

## 9.3 Multi-Head Attention (From Scratch)

Instead of one attention, run **h parallel attention heads** with different projections, then concatenate.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        # Project and reshape: (batch, seq, d_model) → (batch, heads, seq, d_k)
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Attention
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)

        # Concatenate heads: (batch, heads, seq, d_k) → (batch, seq, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_out)

# Test
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, d_model=64
out = mha(x, x, x)  # self-attention
print(f"MHA output: {out.shape}")  # (2, 10, 64)

## 9.4 Positional Encoding

Transformers have no notion of order — positional encoding injects position information.

Uses sine/cosine functions at different frequencies.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

pe = PositionalEncoding(d_model=64)
x = torch.randn(2, 20, 64)
out = pe(x)
print(f"With positional encoding: {out.shape}")

## 9.5 Transformer Block

A single transformer block: **Multi-Head Attention → Add & Norm → FFN → Add & Norm**

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_out = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward with residual connection
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

block = TransformerBlock(d_model=64, num_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
print(f"Block output: {block(x).shape}")  # (2, 10, 64)

## 9.6 Mini GPT — Decoder-Only Transformer

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        mask = create_causal_mask(seq_len).to(x.device)

        x = self.pos_enc(self.token_emb(x))
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        return self.head(x)  # (batch, seq_len, vocab_size)

gpt = MiniGPT(vocab_size=1000, d_model=128, num_heads=4, num_layers=4, d_ff=512)
tokens = torch.randint(0, 1000, (2, 32))
logits = gpt(tokens)
print(f"GPT output: {logits.shape}")  # (2, 32, 1000)

params = sum(p.numel() for p in gpt.parameters())
print(f"Parameters: {params:,}")

## 9.7 Using nn.Transformer (Built-in)

PyTorch provides a ready-made Transformer module.

In [ ]:
transformer = nn.Transformer(
    d_model=64, nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=256,
    batch_first=True
)

src = torch.randn(2, 10, 64)  # encoder input
tgt = torch.randn(2, 15, 64)  # decoder input

# Causal mask for decoder
tgt_mask = nn.Transformer.generate_square_subsequent_mask(15)

out = transformer(src, tgt, tgt_mask=tgt_mask)
print(f"Transformer output: {out.shape}")  # (2, 15, 64)

## 9.8 Hugging Face Integration

For production use, leverage pretrained transformers from Hugging Face.

In [ ]:
# pip install transformers
# from transformers import AutoModel, AutoTokenizer
#
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# model = AutoModel.from_pretrained('bert-base-uncased')
#
# inputs = tokenizer("Hello world!", return_tensors='pt')
# outputs = model(**inputs)
# print(f"Last hidden: {outputs.last_hidden_state.shape}")

print("Uncomment above after: pip install transformers")

## ✅ Phase 9 Summary

You now understand:
- Scaled dot-product attention mechanism
- Multi-head attention for parallel attention patterns
- Positional encoding to inject sequence order
- Transformer blocks (attention + FFN + residuals + norms)
- Decoder-only (GPT) architecture
- PyTorch's built-in nn.Transformer

**Next → Phase 10: Advanced Training Techniques**